# Feature Engineering

`result1/run_honest_local_refit.py`의 입력 준비와 feature candidate 생성 부분을 notebook용으로 분리했습니다.
이 notebook은 modeling notebook이 읽을 전용 artifact를 `notebooks/artifacts/honest_local_refit/` 아래에 저장합니다.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


In [2]:
PROJECT_ROOT = Path("/Users/hyun/workspace/data")
RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACT_DIR = PROJECT_ROOT / "notebooks" / "artifacts" / "honest_local_refit"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SOURCE_CSV = RESULTS_DIR / "batch1_train_batch2_test_full_q1_q5_elasticnet_train_features.csv"
TEST_SOURCE_CSV = RESULTS_DIR / "batch1_train_batch2_test_full_q1_q5_elasticnet_test_features.csv"

BATCH1_ARTIFACT_CSV = ARTIFACT_DIR / "batch1_features.csv"
BATCH2_ARTIFACT_CSV = ARTIFACT_DIR / "batch2_features.csv"
TRAIN_SUB_ARTIFACT_CSV = ARTIFACT_DIR / "train_sub.csv"
VALID_SUB_ARTIFACT_CSV = ARTIFACT_DIR / "valid_sub.csv"
CANDIDATE_SPECS_JSON = ARTIFACT_DIR / "candidate_specs.json"
FEATURE_CONFIG_JSON = ARTIFACT_DIR / "feature_config.json"

BASE_FEATURES = [
    "Qd_delta_100_10",
    "Qd_retention_100_10",
    "IR_delta_100_10",
    "QC_retention_100_10",
    "chargetime_100_mean",
    "IR_cv_1_100",
    "Qd_QC_ratio_100",
    "Qd_slope_1_100",
    "Qd_drop_per_100",
]

EXTRA_POOL = [
    "policy_soc_pct",
    "policy_charge_c_last",
    "Qd_cv_1_100",
    "Qd_resid_std_1_100",
    "IR_100_mean",
    "Qd_mean_1_100",
    "coulombic_efficiency_100_mean",
    "QC_delta_100_10",
    "Qd_100",
]


In [3]:
batch1 = pd.read_csv(TRAIN_SOURCE_CSV)
batch2 = pd.read_csv(TEST_SOURCE_CSV)

batch1 = batch1[batch1["cycle_life"].notna()].copy().reset_index(drop=True)
batch2 = batch2[batch2["cycle_life"].notna()].copy().reset_index(drop=True)

feature_columns = list(dict.fromkeys(BASE_FEATURES + EXTRA_POOL))
required_columns = ["cell_id", "cycle_life", "charging_policy", *feature_columns]

batch1 = batch1[required_columns].copy()
batch2 = batch2[required_columns].copy()

batch1 = batch1.replace([np.inf, -np.inf], np.nan)
batch2 = batch2.replace([np.inf, -np.inf], np.nan)

batch1.shape, batch2.shape

((46, 21), (39, 21))

In [4]:
groups = batch1["charging_policy"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(batch1, groups=groups))

train_sub = batch1.iloc[train_idx].copy().reset_index(drop=True)
valid_sub = batch1.iloc[valid_idx].copy().reset_index(drop=True)

train_sub.shape, valid_sub.shape

((35, 21), (11, 21))

In [5]:
candidate_specs = []
seen = set()


def add_candidate(name: str, features: list[str]) -> None:
    key = tuple(sorted(features))
    if key in seen:
        return
    seen.add(key)
    candidate_specs.append({
        "candidate_name": name,
        "features": list(features),
        "feature_count": len(features),
    })


add_candidate("base", BASE_FEATURES)
for feature in BASE_FEATURES:
    add_candidate(f"remove:{feature}", [f for f in BASE_FEATURES if f != feature])
for feature in EXTRA_POOL:
    add_candidate(f"add:{feature}", BASE_FEATURES + [feature])
for remove_feature in BASE_FEATURES:
    for add_feature in EXTRA_POOL:
        proposal = [f for f in BASE_FEATURES if f != remove_feature] + [add_feature]
        add_candidate(f"swap:{remove_feature}->{add_feature}", proposal)

candidate_specs_df = pd.DataFrame(candidate_specs)
candidate_specs_df.head()

,candidate_name,features,feature_count
0,base,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9
1,remove:Qd_delta_100_10,"[Qd_retention_100_10, IR_delta_100_10, QC_rete...",8
2,remove:Qd_retention_100_10,"[Qd_delta_100_10, IR_delta_100_10, QC_retentio...",8
3,remove:IR_delta_100_10,"[Qd_delta_100_10, Qd_retention_100_10, QC_rete...",8
4,remove:QC_retention_100_10,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",8


In [6]:
batch1.to_csv(BATCH1_ARTIFACT_CSV, index=False)
batch2.to_csv(BATCH2_ARTIFACT_CSV, index=False)
train_sub.to_csv(TRAIN_SUB_ARTIFACT_CSV, index=False)
valid_sub.to_csv(VALID_SUB_ARTIFACT_CSV, index=False)

CANDIDATE_SPECS_JSON.write_text(
    json.dumps(candidate_specs, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

FEATURE_CONFIG_JSON.write_text(
    json.dumps(
        {
            "base_features": BASE_FEATURES,
            "extra_pool": EXTRA_POOL,
            "feature_columns": feature_columns,
            "artifact_dir": str(ARTIFACT_DIR),
            "batch1_rows": int(len(batch1)),
            "batch2_rows": int(len(batch2)),
            "train_sub_rows": int(len(train_sub)),
            "valid_sub_rows": int(len(valid_sub)),
            "candidate_count": int(len(candidate_specs)),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

pd.Series({
    "artifact_dir": str(ARTIFACT_DIR),
    "batch1_rows": len(batch1),
    "batch2_rows": len(batch2),
    "train_sub_rows": len(train_sub),
    "valid_sub_rows": len(valid_sub),
    "candidate_count": len(candidate_specs),
})

artifact_dir       /Users/hyun/workspace/data/notebooks/artifacts...
batch1_rows                                                       46
batch2_rows                                                       39
train_sub_rows                                                    35
valid_sub_rows                                                    11
candidate_count                                                  100
dtype: object